In [1]:
from time import time

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

categories = None


def size_mb(docs):
    return sum(len(s.encode("utf-8")) for s in docs) / 1e6


def load_dataset(verbose=False, remove=()):
    """Load and vectorize the 20 newsgroups dataset."""

    data_train = fetch_20newsgroups(
        subset="train",
        categories=categories,
        shuffle=True,
        random_state=42,
        remove=remove,
    )

    data_test = fetch_20newsgroups(
        subset="test",
        categories=categories,
        shuffle=True,
        random_state=42,
        remove=remove,
    )

    # order of labels in `target_names` can be different from `categories`
    target_names = data_train.target_names

    # split target in a training set and a test set
    y_train, y_test = data_train.target, data_test.target

    # Extracting features from the training data using a sparse vectorizer
    t0 = time()
    vectorizer = TfidfVectorizer(
        sublinear_tf=True, max_df=0.5, min_df=5, stop_words="english"
    )
    X_train = vectorizer.fit_transform(data_train.data)
    duration_train = time() - t0

    # Extracting features from the test data using the same vectorizer
    t0 = time()
    X_test = vectorizer.transform(data_test.data)
    duration_test = time() - t0

    feature_names = vectorizer.get_feature_names_out()

    # Feature type 1: raw word counts (no normalisation, no idf)
    counts_vectorizer = CountVectorizer(max_df=0.5, min_df=5, stop_words="english")
    counts_X_train = counts_vectorizer.fit_transform(data_train.data)
    counts_X_test = counts_vectorizer.transform(data_test.data)

    # Feature type 2: tf only (tf-idf with the idf part switched off).
    # sublinear_tf matches the tf-idf vectorizer above, so idf is the only difference.
    tf_vectorizer = TfidfVectorizer(
        use_idf=False, sublinear_tf=True, max_df=0.5, min_df=5, stop_words="english"
    )
    tf_X_train = tf_vectorizer.fit_transform(data_train.data)
    tf_X_test = tf_vectorizer.transform(data_test.data)

    # Feature type 3 (tf-idf) is X_train / X_test, built above.

    if verbose:
        # compute size of loaded data
        data_train_size_mb = size_mb(data_train.data)
        data_test_size_mb = size_mb(data_test.data)

        print(
            f"{len(data_train.data)} documents - "
            f"{data_train_size_mb:.2f}MB (training set)"
        )
        print(f"{len(data_test.data)} documents - {data_test_size_mb:.2f}MB (test set)")
        print(f"{len(target_names)} categories")
        print(
            f"vectorize training done in {duration_train:.3f}s "
            f"at {data_train_size_mb / duration_train:.3f}MB/s"
        )
        print(f"n_samples: {X_train.shape[0]}, n_features: {X_train.shape[1]}")
        print(
            f"vectorize testing done in {duration_test:.3f}s "
            f"at {data_test_size_mb / duration_test:.3f}MB/s"
        )
        print(f"n_samples: {X_test.shape[0]}, n_features: {X_test.shape[1]}")

    return (
        X_train,
        X_test,
        y_train,
        y_test,
        feature_names,
        target_names,
        tf_X_train,
        tf_X_test,
        counts_X_train,
        counts_X_test,
    )

In [ ]:
X_train, X_test, y_train, y_test, feature_names, target_names, *_ = load_dataset()
from sklearn.linear_model import RidgeClassifier

clf = RidgeClassifier(tol=1e-2, solver="sparse_cg")
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(20, 10))
fig.set_dpi(300)
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, cmap=plt.cm.Blues)
ax.xaxis.set_ticklabels(target_names, rotation=45, ha="right")
ax.yaxis.set_ticklabels(target_names)
_ = ax.set_title(
    f"Confusion Matrix for {clf.__class__.__name__}\non the original documents"
)

In [ ]:
import numpy as np
import pandas as pd


def plot_feature_effects():
    # learned coefficients weighted by frequency of appearance
    average_feature_effects = clf.coef_ * np.asarray(X_train.mean(axis=0)).ravel()

    for i, label in enumerate(target_names):
        top5 = np.argsort(average_feature_effects[i])[-5:][::-1]
        if i == 0:
            top = pd.DataFrame(feature_names[top5], columns=[label])
            top_indices = top5
        else:
            top[label] = feature_names[top5]
            top_indices = np.concatenate((top_indices, top5), axis=None)
    top_indices = np.unique(top_indices)
    predictive_words = feature_names[top_indices]

    # plot feature effects
    bar_size = 0.25
    padding = 0.75
    y_locs = np.arange(len(top_indices)) * (20 * bar_size + padding)

    fig, ax = plt.subplots(figsize=(20, 20))
    fig.set_dpi(300)
    for i, label in enumerate(target_names):
        ax.barh(
            y_locs + (i - 2) * bar_size,
            average_feature_effects[i, top_indices],
            height=bar_size,
            label=label,
        )
    ax.set(
        yticks=y_locs,
        yticklabels=predictive_words,
        ylim=[
            0 - 20 * bar_size,
            len(top_indices) * (20 * bar_size + padding) - 20 * bar_size,
        ],
    )
    ax.legend(loc="lower right")

    print("top 5 keywords per class:")
    print(top)

    return ax


_ = plot_feature_effects().set_title("Average feature effect on the original data")

In [ ]:
data_train = fetch_20newsgroups(
    subset="train", categories=categories, shuffle=True, random_state=42
)

for doc in data_train.data:
    if "caltech" in doc:
        print(doc)
        break

In [ ]:
(
    X_train,
    X_test,
    y_train,
    y_test,
    feature_names,
    target_names,
    tf_X_train,
    tf_X_test,
    counts_X_train,
    counts_X_test,
) = load_dataset(remove=("headers", "footers", "quotes"))

# the tf-idf matrices are X_train / X_test; give them an explicit name too
tfidf_X_train, tfidf_X_test = X_train, X_test

clf = RidgeClassifier(tol=1e-2, solver="sparse_cg")
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

fig, ax = plt.subplots(figsize=(15, 10))
fig.set_dpi(300)
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, cmap= plt.cm.Blues)
ax.xaxis.set_ticklabels(target_names, rotation=45, ha="right")
ax.yaxis.set_ticklabels(target_names)
_ = ax.set_title(
    f"Confusion Matrix for {clf.__class__.__name__}\non filtered documents"
)

In [ ]:
_ = plot_feature_effects().set_title("Average feature effects on filtered documents")

In [ ]:
from sklearn import metrics
from sklearn.utils.extmath import density


def benchmark(clf, X_train, X_test, y_train, y_test, custom_name=False):
    """Train clf on (X_train, y_train) and evaluate it on (X_test, y_test)."""
    print("_" * 80)
    print("Training: ")
    print(clf)
    t0 = time()
    clf.fit(X_train, y_train)
    train_time = time() - t0
    print(f"train time: {train_time:.3}s")

    t0 = time()
    pred = clf.predict(X_test)
    test_time = time() - t0
    print(f"test time:  {test_time:.3}s")

    score = metrics.accuracy_score(y_test, pred)
    print(f"accuracy:   {score:.3}")

    if hasattr(clf, "coef_"):
        print(f"dimensionality: {clf.coef_.shape[1]}")
        print(f"density: {density(clf.coef_)}")
        print()

    print()
    if custom_name:
        clf_descr = str(custom_name)
    else:
        clf_descr = clf.__class__.__name__
    return clf_descr, score, train_time, test_time


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.svm import LinearSVC

# the three feature types: (train matrix, test matrix)
feature_sets = {
    "counts": (counts_X_train, counts_X_test),
    "tf": (tf_X_train, tf_X_test),
    "tf-idf": (tfidf_X_train, tfidf_X_test),
}


def make_classifiers():
    """Fresh, untrained classifiers (so nothing is reused between feature types)."""
    return (
        (LogisticRegression(C=5, max_iter=1000), "Logistic Regression"),
        (RidgeClassifier(alpha=1.0, solver="sparse_cg"), "Ridge Classifier"),
        (
            KNeighborsClassifier(n_neighbors=15, algorithm="brute", metric="cosine"),
            "kNN",
        ),
        (RandomForestClassifier(), "Random Forest"),
        # L2 penalty Linear SVC
        (LinearSVC(C=0.1, dual=False, max_iter=1000), "Linear SVC"),
        # L2 penalty Linear SGD
        (
            SGDClassifier(
                loss="log_loss", alpha=1e-4, n_iter_no_change=3, early_stopping=True
            ),
            "log-loss SGD",
        ),
        # NearestCentroid (aka Rocchio classifier)
        (NearestCentroid(), "NearestCentroid"),
        # Sparse naive Bayes classifier
        (ComplementNB(alpha=0.1), "Complement naive Bayes"),
    )


# Benchmark run 1: full per-model printout, every classifier on every feature type
results_all = []
for feature_name, (Xtr, Xte) in feature_sets.items():
    print("#" * 80)
    print(f"FEATURES: {feature_name}")
    for clf, name in make_classifiers():
        print("=" * 80)
        print(name)
        clf_descr, score, train_time, test_time = benchmark(clf, Xtr, Xte, y_train, y_test, name)
        results_all.append(
            {
                "features": feature_name,
                "classifier": clf_descr,
                "accuracy": score,
                "train_time": train_time,
                "test_time": test_time,
            }
        )


In [ ]:
# Benchmark run 2: compact summary of the results above (no retraining)
summary = pd.DataFrame(results_all)

print("Test accuracy (rows: classifier, columns: feature type)")
print(summary.pivot(index="classifier", columns="features", values="accuracy").round(3))
print()
print("Training time in seconds")
print(summary.pivot(index="classifier", columns="features", values="train_time").round(2))
print()

best = summary.loc[summary["accuracy"].idxmax()]
print(
    f"Best combination by test accuracy: {best['classifier']} + {best['features']} "
    f"(accuracy {best['accuracy']:.3f}, train time {best['train_time']:.2f}s)"
)
print()
print("Top 5 combinations:")
print(summary.sort_values("accuracy", ascending=False).head(5).round(3).to_string(index=False))


In [ ]:
# Trade-off plots for the tf-idf features (as in the original tutorial)
tfidf_res = summary[summary["features"] == "tf-idf"]
clf_names = tfidf_res["classifier"].tolist()
score = tfidf_res["accuracy"].to_numpy()
training_time = tfidf_res["train_time"].to_numpy()
test_time = tfidf_res["test_time"].to_numpy()

fig, ax1 = plt.subplots(figsize=(10, 8))
ax1.scatter(score, training_time, s=60)
ax1.set(
    title="Score-training time trade-off (tf-idf)",
    yscale="log",
    xlabel="test accuracy",
    ylabel="training time (s)",
)
fig, ax2 = plt.subplots(figsize=(10, 8))
ax2.scatter(score, test_time, s=60)
ax2.set(
    title="Score-test time trade-off (tf-idf)",
    yscale="log",
    xlabel="test accuracy",
    ylabel="test time (s)",
)

for i, txt in enumerate(clf_names):
    ax1.annotate(txt, (score[i], training_time[i]))
    ax2.annotate(txt, (score[i], test_time[i]))


## Vectorizer parameter grid (tf-idf + Complement naive Bayes)

Every combination of lowercasing, stop words, analyzer/n-gram range and `max_features` is tried.
Combinations are compared on a **validation split of the training data**; the test set is used
only once, for the winning combination.

In [ ]:
import itertools

from sklearn.model_selection import train_test_split

# Raw texts (the vectorizer is now part of the experiment, so we need the text, not matrices)
raw_train = fetch_20newsgroups(
    subset="train", shuffle=True, random_state=42,
    remove=("headers", "footers", "quotes"),
)
raw_test = fetch_20newsgroups(
    subset="test", shuffle=True, random_state=42,
    remove=("headers", "footers", "quotes"),
)

# validation split of the training data, used to compare the combinations
texts_fit, texts_val, y_fit, y_val = train_test_split(
    raw_train.data, raw_train.target,
    test_size=0.2, stratify=raw_train.target, random_state=42,
)

# ---- the parameter values to try (one array per parameter) ----
lowercase_values = [True, False]
stop_words_values = [None, "english"]
analyzer_ngram_values = [          # analyzer and ngram_range belong together
    ("word", (1, 1)),
    ("word", (1, 2)),
    ("char_wb", (3, 5)),
]
max_features_values = [1000, 5000, 20000, None]

# ---- the array with all combinations ----
param_combinations = []
for lowercase, stop_words, (analyzer, ngram_range), max_features in itertools.product(
    lowercase_values, stop_words_values, analyzer_ngram_values, max_features_values
):
    # stop_words is only used by the "word" analyzer; for character n-grams the
    # "english" variant would give the identical result, so we skip those duplicates
    if analyzer != "word" and stop_words is not None:
        continue
    param_combinations.append(
        {
            "lowercase": lowercase,
            "stop_words": stop_words,
            "analyzer": analyzer,
            "ngram_range": ngram_range,
            "max_features": max_features,
        }
    )

print(len(param_combinations), "combinations")

In [ ]:
def run_vectorizer_experiments(param_combinations, train_texts, train_y, eval_texts, eval_y):
    """For every parameter combination: fit TfidfVectorizer on train_texts, train
    ComplementNB, and measure accuracy on eval_texts. Returns a DataFrame with one row
    per combination (parameters, number of features, accuracy, timings)."""
    rows = []
    for i, params in enumerate(param_combinations, start=1):
        t0 = time()
        vectorizer = TfidfVectorizer(
            sublinear_tf=True,   # same fixed settings as the earlier experiments
            max_df=0.5,
            min_df=5,
            **params,
        )
        X_tr = vectorizer.fit_transform(train_texts)
        X_ev = vectorizer.transform(eval_texts)
        vectorize_time = time() - t0

        t0 = time()
        model = ComplementNB(alpha=0.1)
        model.fit(X_tr, train_y)
        accuracy = metrics.accuracy_score(eval_y, model.predict(X_ev))
        classify_time = time() - t0

        rows.append(
            {
                **params,
                "n_features": X_tr.shape[1],
                "accuracy": accuracy,
                "vectorize_time": vectorize_time,
                "classify_time": classify_time,
            }
        )
        print(f"[{i}/{len(param_combinations)}] {params} -> accuracy {accuracy:.3f}")
    return pd.DataFrame(rows)

In [ ]:
grid_results = run_vectorizer_experiments(
    param_combinations, texts_fit, y_fit, texts_val, y_val
)
grid_results.to_csv("vectorizer_param_results.csv", index=False)   # stored results

print()
print("Top 10 combinations (validation accuracy):")
print(grid_results.sort_values("accuracy", ascending=False).head(10).round(3).to_string(index=False))

In [ ]:
# Effect of each parameter on its own: mean validation accuracy per value
grid_results["analyzer + ngram"] = (
    grid_results["analyzer"] + " " + grid_results["ngram_range"].astype(str)
)
for column in ["lowercase", "stop_words", "analyzer + ngram", "max_features"]:
    print(f"--- {column}")
    print(
        grid_results.fillna({"stop_words": "none", "max_features": "all"})
        .groupby(column)["accuracy"].agg(["mean", "max"]).round(3)
    )
    print()

# Winner on the validation split, scored once on the real test set
best = grid_results.loc[grid_results["accuracy"].idxmax()]
best_params = {
    k: best[k] for k in ["lowercase", "stop_words", "analyzer", "ngram_range", "max_features"]
}
best_params["stop_words"] = None if pd.isna(best_params["stop_words"]) else best_params["stop_words"]
best_params["max_features"] = None if pd.isna(best_params["max_features"]) else int(best_params["max_features"])
print("Best parameters:", best_params, f"(validation accuracy {best['accuracy']:.3f})")

final_vectorizer = TfidfVectorizer(sublinear_tf=True, max_df=0.5, min_df=5, **best_params)
X_final_train = final_vectorizer.fit_transform(raw_train.data)       # full training set
X_final_test = final_vectorizer.transform(raw_test.data)
final_model = ComplementNB(alpha=0.1).fit(X_final_train, raw_train.target)
print("Test accuracy of the best combination:",
      round(metrics.accuracy_score(raw_test.target, final_model.predict(X_final_test)), 3))